<a href="https://colab.research.google.com/github/iras-mpark/MLA1020/blob/main/week6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import numpy as np
import torch
from typing import Any


In [13]:
class Step:
  """Represents taking an `action`, incurring some `cost` and ending up in a new `state`."""
  def __init__(self, action: Any, cost: float, state: Any):
    self.action = action
    self.cost = cost
    self.state = state

In [14]:
class SearchProblem:
  """Formally and fully represents a search problem."""
  def start_state(self) -> Any:
    raise NotImplementedError
  def successors(self, state: Any) -> list[Step]:
    raise NotImplementedError
  def is_end(self, state: Any) -> bool:
    raise NotImplementedError

In [15]:
class Solution:
  """Represents a solution to a search problem (sequence of actions that produces a cost)."""
  steps: list[Step]
  cost: float
  def __init__(self, steps: list[Step]):
    self.steps = steps
    # The cost of a solution is the sum of the costs of the actions
    costs = [step.cost for step in steps]
    self.cost = sum(costs)

# UCS Example

### Diamond problem formulation

In [16]:
class DiamondSearchProblem(SearchProblem):
  def __init__(self):
    # state -> state -> cost
    self.graph = {
      "A": {"B": 1, "C": 100},
      "B": {"A": 1, "C": 1, "D": 100},
      "C": {"A": 100, "B": 1, "D": 1},
      "D": {"B": 100, "C": 1},
    }
  def start_state(self) -> str:
    return "A"

  def successors(self, state: str) -> list[Step]:
    return [
      Step(action=new_state, cost=cost, state=new_state) \
      for new_state, cost in self.graph[state].items()
    ]

  def is_end(self, state: str) -> bool:
    return state == "D"

# UCS Algorithm

In [17]:
class Backpointer:
  """For a state, record how we got to it."""
  def __init__(self, prev_state: Any, action: float, cost: float):
    self.prev_state = prev_state # Which state we came from
    self.action = action # Action we took from `prev_state`
    self.cost = cost # Cost of that action

In [18]:
import heapq

class PriorityQueue:
  """Data structure for supporting uniform cost search."""
  def __init__(self):
    self.DONE = -100000
    self.heap = []
    self.priorities = {}  # Map from state to priority
  def update(self, state: Any, new_priority: float) -> bool:
    """
    Insert `state` into the heap with priority `new_priority` if `state`
    isn't in the heap or `new_priority` is smaller than the existing
    priority.  Return whether the priority queue was updated.
    """
    old_priority = self.priorities.get(state)
    if old_priority is None or new_priority < old_priority:
      self.priorities[state] = new_priority
      heapq.heappush(self.heap, (new_priority, state))
      return True
    return False
  def remove_min(self):
    """Return (state with minimum priority, priority) or (None, None) if empty."""
    while len(self.heap) > 0:
      priority, state = heapq.heappop(self.heap)
      if self.priorities[state] == self.DONE:
        # Outdated priority, skip
        continue
      self.priorities[state] = self.DONE
      return state, priority
    # Nothing left...
    return None, None

In [19]:
def uniform_cost_search(problem: SearchProblem) -> tuple[Solution | None, int]:
  """
  Run Uniform Cost Search (UCS) on the specified search `problem`.
  Return the solution (sequence of steps) and the number of states explored.
  """
  # Frontier: states we've seen, still trying to figure out how the best way to get there
  # Priority represents the minimum cost to get there
  frontier = PriorityQueue()
  # For each state we've reached, backpointer tells us how we got there
  backpointers: dict[Any, Backpointer] = {}
  num_explored = 0
  # Add the start state
  start_state = problem.start_state()
  frontier.update(start_state, 0.0)
  while True:
    # Remove the state from the frontier with the lowest priority (theorem: priority = past_cost).
    state, past_cost = frontier.remove_min()
    if state is None and past_cost is None:
      return None, num_explored  # Found no solution
    num_explored += 1
    # Check if we've reached an end state; if so, extract solution.
    if problem.is_end(state):
      # Walk back the backpointers to get the actions
      steps = []
      while state != start_state:
        backpointer = backpointers[state]
        steps.insert(0, Step(backpointer.action, backpointer.cost, state))  # Prepend
        state = backpointer.prev_state  # Go back
      return Solution(steps=steps), num_explored
    # Expand from `state`, updating the frontier with each `new_state`
    for successor in problem.successors(state):
      if frontier.update(successor.state, past_cost + successor.cost):
          # We found better way to get to `successor.state` --> update backpointer!
          backpointers[successor.state] = Backpointer(prev_state=state, action=successor.action, cost=successor.cost)

In [20]:
problem = DiamondSearchProblem()
solution, num_explored = uniform_cost_search(problem)

In [21]:
print("Total state changes", len(solution.steps))

path_str = str(problem.start_state())
for step in solution.steps:
    path_str += " → " + str(step.state)
print(path_str)
print("Total cost:", solution.cost)

Total state changes 3
A → B → C → D
Total cost: 3


# A-star Examples

### Grid Search

In [22]:
class GridSearchProblem(SearchProblem):
  def __init__(self, *rows: list[str]):
    # Remove spaces (which are just for readability)
    self.rows = [row.replace(" ", "") for row in rows]
  def start_state(self) -> str:
    for r in range(self.num_rows):
      for c in range(self.num_cols):
        if self.rows[r][c] == "S":
          return (r, c)
    raise ValueError("No start state found")
  def is_valid(self, state: tuple[int, int]) -> bool:
    """Check if a state is valid."""
    r, c = state
    return 0 <= r < self.num_rows and \
            0 <= c < self.num_cols and \
            self.rows[r][c] != "#"

  def successors(self, state: tuple[int, int]) -> list[Step]:
    """Return the successors of a state (up, down, left, right)."""
    r, c = state
    successors = []
    if self.is_valid((r + 1, c)):
      successors.append(Step(action="down", cost=1, state=(r + 1, c)))
    if self.is_valid((r - 1, c)):
      successors.append(Step(action="up", cost=1, state=(r - 1, c)))
    if self.is_valid((r, c + 1)):
      successors.append(Step(action="right", cost=1, state=(r, c + 1)))
    if self.is_valid((r, c - 1)):
      successors.append(Step(action="left", cost=1, state=(r, c - 1)))
    return successors
  def is_end(self, state: tuple[int, int]) -> bool:
    """Check if a state is an end state."""
    r, c = state
    return self.rows[r][c] == "E"